###IMPORTING NEEDED LIBRARIES FOR THE PROJECT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

print("Libraries imported successfully!")

###IMPORT DATASETS FOR THE PROJECT

In [ ]:
df=pd.read_csv("/content/movies.csv")
df


In [ ]:
file_path = '/content/combined_data_1.txt'

with open(file_path, 'r') as file:
    for i in range(15):
        print(file.readline().strip())

###DOING EDA

In [ ]:
df.info()

In [ ]:
#df.isnull().sum()
df.isna().sum()

In [ ]:
import pandas as pd
import random

sample_size = 200000

sample = []
movie_id = None
count = 0

random.seed(42)

with open('/content/combined_data_1.txt', 'r') as file:

    for line in file:

        line = line.strip()

        if line.endswith(':'):
            movie_id = int(line[:-1])

        else:
            user_id, rating, date = line.split(',')

            record = (
                int(user_id),
                movie_id,
                int(rating),
                date
            )

            count += 1

            if len(sample) < sample_size:
                sample.append(record)

            else:
                index = random.randint(0, count - 1)

                if index < sample_size:
                    sample[index] = record

ratings = pd.DataFrame(
    sample,
    columns=['User_ID', 'Movie_ID', 'Rating', 'Date']
)

display(ratings.head())
print("Dataset Shape:", ratings.shape)

In [ ]:
print("Dataset Shape:", ratings.shape)

print("\nColumn Names:")
print(ratings.columns.tolist())

print("\nData Types:")
print(ratings.dtypes)

print("\nDataset Information:")
ratings.info()

In [ ]:
ratings.isnull().sum()

In [ ]:
ratings.duplicated().sum()

In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=ratings,
    x='Rating',
    color='blue'
)

plt.title('Distribution of Movie Ratings')
plt.xlabel('Movie Rating')
plt.ylabel('Number of Ratings')

plt.show()

In [ ]:
popular_movies = (
    ratings.groupby('Movie_ID')['Rating']
    .count()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Most Rated Movies:")
print(popular_movies)

In [ ]:
plt.figure(figsize=(12, 6))

popular_movies.plot(
    kind='bar',
    color='steelblue'
)

plt.title('Top 10 Most Rated Movies')
plt.xlabel('Movie ID')
plt.ylabel('Number of Ratings')

plt.xticks(rotation=45)

plt.show()

In [ ]:
movie_analysis = ratings.groupby('Movie_ID').agg(
    Average_Rating=('Rating', 'mean'),
    Total_Ratings=('Rating', 'count')
)

# Consider movies with at least 50 ratings
top_rated_movies = (
    movie_analysis[
        movie_analysis['Total_Ratings'] >= 50
    ]
    .sort_values(
        by='Average_Rating',
        ascending=False
    )
    .head(10)
)

print(top_rated_movies)

In [ ]:
user_activity=ratings.groupby('User_ID')['Rating'].count()
user_activity

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    user_activity,
    bins=30,
    color='orange',
    edgecolor='black'
)

plt.title('Distribution of User Activity')
plt.xlabel('Number of Movies Rated')
plt.ylabel('Number of Users')

plt.show()

In [ ]:
ratings['Date'] = pd.to_datetime(ratings['Date'])

ratings['Year'] = ratings['Date'].dt.year

yearly_ratings = ratings.groupby('Year').size()

plt.figure(figsize=(12, 5))

yearly_ratings.plot(
    kind='line',
    marker='o'
)

plt.title('Movie Rating Activity Over Years')
plt.xlabel('Year')
plt.ylabel('Number of Ratings')

plt.show()

In [ ]:
genre_data = df['genres'].str.split('|').explode()

genre_counts = genre_data.value_counts()

print("Top 15 Movie Genres:")
print(genre_counts.head(15))

In [ ]:
plt.figure(figsize=(12, 6))

genre_counts.head(15).plot(
    kind='bar',
    color='teal'
)

plt.title('Top 15 Movie Genres')
plt.xlabel('Genre')
plt.ylabel('Number of Movies')

plt.xticks(rotation=45)

plt.show()

In [ ]:
ratings.to_csv(
    '/content/ratings_after_eda.csv',
    index=False
)

movie_analysis.to_csv(
    '/content/movie_analysis.csv'
)

user_activity.to_csv(
    '/content/user_activity.csv'
)

print("EDA results saved successfully!")

In [ ]:

# Count unique users and movies

unique_users = ratings['User_ID'].nunique()
unique_movies = ratings['Movie_ID'].nunique()

print("Total Ratings:", len(ratings))
print("Unique Users:", unique_users)
print("Unique Movies:", unique_movies)

print("\nUser-Movie Statistics:")
print(ratings[['User_ID', 'Movie_ID', 'Rating']].describe())


In [ ]:

# Number of ratings given by each user

user_ratings = ratings.groupby('User_ID')['Rating'].count()

# Number of ratings received by each movie

movie_ratings = ratings.groupby('Movie_ID')['Rating'].count()

print("User Rating Statistics:")
print(user_ratings.describe())

print("\nMovie Rating Statistics:")
print(movie_ratings.describe())


In [ ]:

# Minimum interaction threshold

min_user_ratings = 3
min_movie_ratings = 3

# Count ratings per user and movie

user_counts = ratings['User_ID'].value_counts()
movie_counts = ratings['Movie_ID'].value_counts()

# Keep users and movies meeting the threshold

ratings_filtered = ratings[
    (ratings['User_ID'].isin(user_counts[user_counts >= min_user_ratings].index)) &
    (ratings['Movie_ID'].isin(movie_counts[movie_counts >= min_movie_ratings].index))
].copy()

print("Original Dataset Shape:", ratings.shape)
print("Filtered Dataset Shape:", ratings_filtered.shape)

print("\nRemaining Unique Users:", ratings_filtered['User_ID'].nunique())
print("Remaining Unique Movies:", ratings_filtered['Movie_ID'].nunique())


In [ ]:

# Select required columns

ratings_model = ratings_filtered[
    ['User_ID', 'Movie_ID', 'Rating']
].copy()

print(ratings_model.head())

print("\nFinal Model Dataset Shape:")
print(ratings_model.shape)


In [ ]:

!pip install scikit-surprise


In [ ]:

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

print("Surprise library imported successfully!")


In [ ]:

# Define rating scale

reader = Reader(rating_scale=(1, 5))

# Convert Pandas DataFrame into Surprise Dataset

data = Dataset.load_from_df(
    ratings_model[['User_ID', 'Movie_ID', 'Rating']],
    reader
)

print("Dataset successfully converted for SVD!")


In [ ]:

trainset, testset = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

print("Training Records:", trainset.n_ratings)
print("Testing Records:", len(testset))


In [ ]:

# Initialize the SVD model

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

print("SVD Model Initialized Successfully!")


In [ ]:
svd_model.fit(trainset)

In [ ]:

# Generate predictions on testing data

predictions = svd_model.test(testset)

# Display first 10 predictions

for prediction in predictions[:10]:
    print(
        "User:", prediction.uid,
        "| Movie:", prediction.iid,
        "| Actual Rating:", prediction.r_ui,
        "| Predicted Rating:", round(prediction.est, 2)
    )


In [ ]:

# Calculate RMSE

rmse = accuracy.rmse(predictions)

print("Final RMSE:", round(rmse, 4))


In [ ]:

# Calculate MAE

mae = accuracy.mae(predictions)

print("Final MAE:", round(mae, 4))


In [ ]:

# Model evaluation summary

evaluation_results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE'],
    'Score': [rmse, mae]
})

display(evaluation_results)


In [ ]:

# Select a sample user

sample_user = ratings_model['User_ID'].iloc[0]

print("Selected User ID:", sample_user)


In [ ]:

# Movies already rated by the selected user

watched_movies = ratings_model[
    ratings_model['User_ID'] == sample_user
]['Movie_ID'].unique()

print("Movies Already Rated:", len(watched_movies))


In [ ]:

# Get all available movie IDs in the model dataset

all_movies = ratings_model['Movie_ID'].unique()

# Find movies not rated by the selected user

unrated_movies = [
    movie for movie in all_movies
    if movie not in watched_movies
]

# Predict ratings

recommendations = []

for movie_id in unrated_movies:
    predicted_rating = svd_model.predict(
        sample_user,
        movie_id
    ).est

    recommendations.append(
        (movie_id, predicted_rating)
    )

# Sort by predicted rating

recommendations.sort(
    key=lambda x: x[1],
    reverse=True
)

# Display Top 10 recommendations

top_10_recommendations = recommendations[:10]

recommendation_df = pd.DataFrame(
    top_10_recommendations,
    columns=['Movie_ID', 'Predicted_Rating']
)

display(recommendation_df)


In [ ]:

# Convert the existing Surprise training set back into a DataFrame

train_rows = [
    (
        trainset.to_raw_uid(user_id),
        trainset.to_raw_iid(movie_id),
        rating
    )
    for user_id, movie_id, rating in trainset.all_ratings()
]

train_df = pd.DataFrame(
    train_rows,
    columns=['User_ID', 'Movie_ID', 'Rating']
)

print("Training Dataset Shape:", train_df.shape)
display(train_df.head())


In [ ]:

# Convert training DataFrame into Surprise format

train_data = Dataset.load_from_df(
    train_df[['User_ID', 'Movie_ID', 'Rating']],
    reader
)

print("Training data prepared for hyperparameter tuning!")


In [ ]:

from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.005],
    'reg_all': [0.02, 0.05]
}

print("Parameter combinations prepared!")


In [ ]:

grid_search = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search.fit(train_data)

print("Best RMSE:", grid_search.best_score['rmse'])
print("Best MAE:", grid_search.best_score['mae'])

print("\nBest Parameters:")
print(grid_search.best_params['rmse'])


In [ ]:

# Get the best parameters

best_params = grid_search.best_params['rmse']

# Initialize optimized model

best_svd_model = SVD(
    **best_params,
    random_state=42
)

# Train optimized model

best_svd_model.fit(trainset)

print("Optimized SVD Model Trained Successfully!")


In [ ]:

# Generate optimized model predictions

best_predictions = best_svd_model.test(testset)

# Calculate evaluation metrics

optimized_rmse = accuracy.rmse(best_predictions)
optimized_mae = accuracy.mae(best_predictions)

print("\nOptimized Model Results")
print("RMSE:", round(optimized_rmse, 4))
print("MAE:", round(optimized_mae, 4))


In [ ]:

comparison = pd.DataFrame({
    'Model': ['Original SVD', 'Optimized SVD'],
    'RMSE': [rmse, optimized_rmse],
    'MAE': [mae, optimized_mae]
})

display(comparison)


In [ ]:

comparison.set_index('Model').plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title('Original vs Optimized SVD Model')
plt.ylabel('Error Score')
plt.xlabel('Model')
plt.xticks(rotation=0)
plt.legend(title='Evaluation Metric')
plt.show()


In [ ]:

# Select the optimized SVD model
final_model = best_svd_model

print("Final Recommendation Model Selected!")


In [ ]:

# Select a user for recommendations

sample_user = ratings_model['User_ID'].iloc[0]

print("Selected User ID:", sample_user)


In [ ]:

# Find movies already rated by the user

watched_movies = ratings_model[
    ratings_model['User_ID'] == sample_user
]['Movie_ID'].unique()

# Find all movies available in the dataset

all_movies = ratings_model['Movie_ID'].unique()

# Identify movies not rated by this user

unrated_movies = [
    movie for movie in all_movies
    if movie not in watched_movies
]

print("Already Rated Movies:", len(watched_movies))
print("Available Unrated Movies:", len(unrated_movies))


In [ ]:

# Generate recommendations

recommendations = []

for movie_id in unrated_movies:

    predicted_rating = final_model.predict(
        sample_user,
        movie_id
    ).est

    recommendations.append({
        'Movie_ID': movie_id,
        'Predicted_Rating': predicted_rating
    })

# Convert recommendations into DataFrame

final_recommendations = pd.DataFrame(recommendations)

# Sort by predicted rating

final_recommendations = final_recommendations.sort_values(
    by='Predicted_Rating',
    ascending=False
)

# Display Top 10 recommendations

top_10 = final_recommendations.head(10)

display(top_10)


In [ ]:

from collections import defaultdict

# Generate predictions on test data

ranking_predictions = final_model.test(testset)

# Group predictions by user

user_predictions = defaultdict(list)

for uid, iid, actual_rating, estimated_rating, details in ranking_predictions:

    user_predictions[uid].append(
        (iid, estimated_rating, actual_rating)
    )

# Calculate Precision@10 and Recall@10

k = 10

precisions = []
recalls = []

for uid, user_ratings_list in user_predictions.items():

    # Sort by predicted rating

    user_ratings_list.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Top K recommendations from the test set

    top_k = user_ratings_list[:k]

    # Relevant movies have actual ratings >= 4

    relevant_items = [
        item for item in user_ratings_list
        if item[2] >= 4
    ]

    relevant_recommended = sum(
        1 for item in top_k
        if item[2] >= 4
    )

    # Calculate precision and recall

    precision = relevant_recommended / k

    if len(relevant_items) > 0:
        recall = relevant_recommended / len(relevant_items)

        precisions.append(precision)
        recalls.append(recall)

# Calculate average results

precision_at_10 = np.mean(precisions)
recall_at_10 = np.mean(recalls)

print("Precision@10:", round(precision_at_10, 4))
print("Recall@10:", round(recall_at_10, 4))


In [ ]:

# Create evaluation metrics DataFrame

metrics = pd.DataFrame({
    'Metric': [
        'RMSE',
        'MAE',
        'Precision@10',
        'Recall@10'
    ],
    'Score': [
        optimized_rmse,
        optimized_mae,
        precision_at_10,
        recall_at_10
    ]
})

display(metrics)

# Plot evaluation results

plt.figure(figsize=(10, 5))

plt.bar(
    metrics['Metric'],
    metrics['Score']
)

plt.title('Netflix Recommendation System Performance')
plt.xlabel('Evaluation Metrics')
plt.ylabel('Score')

plt.show()


In [ ]:

plt.figure(figsize=(12, 6))

plt.barh(
    top_10['Movie_ID'].astype(str),
    top_10['Predicted_Rating']
)

plt.title('Top 10 Personalized Movie Recommendations')
plt.xlabel('Predicted Rating')
plt.ylabel('Movie ID')

plt.gca().invert_yaxis()

plt.show()


In [ ]:

# Save final recommendation results

final_recommendations.to_csv(
    '/content/final_recommendations.csv',
    index=False
)

# Save evaluation metrics

metrics.to_csv(
    '/content/model_evaluation.csv',
    index=False
)

# Save model comparison

comparison.to_csv(
    '/content/model_comparison.csv',
    index=False
)

print("All project results saved successfully!")


# NETFLIX MOVIE RECOMMENDATION ENGINE
## Capstone Project Report

**Project Domain:** Artificial Intelligence and Machine Learning  
**Technology:** Python, Machine Learning, Collaborative Filtering  
**Algorithm:** Singular Value Decomposition (SVD)

---

# 1. Introduction

In today's digital world, online entertainment platforms provide users with thousands of movies and television shows. Finding relevant content among such a large collection can be difficult.

Netflix uses recommendation systems to help users discover movies and shows according to their interests and viewing preferences.

The Netflix Movie Recommendation Engine is a machine learning project designed to predict user preferences and recommend movies based on historical user ratings.

This project implements a collaborative filtering recommendation approach using Singular Value Decomposition (SVD). The algorithm identifies hidden relationships between users and movies by analyzing previously provided ratings.

The primary objective is to develop a recommendation system that predicts ratings for movies a user has not rated and ranks movies according to their predicted preference.

# 2. Problem Statement

Users often face difficulties discovering movies that match their interests because of the enormous amount of available content.

Traditional movie browsing methods require users to manually search through available titles.

The purpose of this project is to develop an intelligent recommendation system that analyzes historical movie ratings, identifies user preferences, predicts unknown ratings and recommends potentially interesting movies.

# 3. Objectives of the Project

The main objectives are:

1. To understand and analyze movie rating datasets.
2. To perform data cleaning and preprocessing.
3. To conduct Exploratory Data Analysis (EDA).
4. To understand relationships between users, movies and ratings.
5. To implement collaborative filtering using SVD.
6. To predict movie ratings for individual users.
7. To evaluate model performance using RMSE and MAE.
8. To improve model performance through hyperparameter tuning.
9. To generate personalized movie recommendations.
10. To visualize and interpret recommendation results.

# 4. Dataset Description

The project uses two datasets:

### A. Netflix Rating Dataset

The `combined_data_1.txt` file contains historical movie rating records.

The dataset follows a structured format in which each movie ID is followed by its associated user ratings.

The rating records contain:

- Movie ID
- User ID
- Rating
- Date

For this project, a random sample of 200,000 rating records was selected for experimentation.

### B. Movie Metadata Dataset

The `movies.csv` file contains movie-related information, including:

- Movie ID
- Movie Title
- Genres

This dataset was used for separate genre analysis.

**Dataset compatibility limitation:** The Netflix rating dataset and the supplied `movies.csv` appear to use different movie identification systems. Therefore, their movie IDs cannot be directly joined without a verified mapping.

# 5. Technologies and Libraries Used

| Technology | Purpose |
|---|---|
| Python | Programming language |
| Google Colab | Development environment |
| Pandas | Data manipulation and analysis |
| NumPy | Numerical computations |
| Matplotlib | Data visualization |
| Seaborn | Statistical visualization |
| Scikit-Surprise | Recommendation algorithm implementation |
| Scikit-Learn | Machine learning utilities |

# 6. Methodology

The project follows a systematic machine learning workflow.

### 6.1 Data Collection

The Netflix rating dataset and movie metadata file were imported into Google Colab.

### 6.2 Data Preprocessing

The rating dataset was parsed to extract movie IDs, user IDs, ratings and dates.

A sample of 200,000 records was created to make experimentation more manageable.

### 6.3 Exploratory Data Analysis

EDA was performed to understand the dataset's characteristics.

The following analyses were conducted:

- Dataset dimensions and data types.
- Missing value identification.
- Duplicate record checking.
- Rating distribution analysis.
- Most-rated movies analysis.
- Highly rated movies analysis.
- User activity analysis.
- Yearly rating trends.
- Movie genre distribution.

### 6.4 Feature Preparation

The dataset was prepared using three essential columns:

- User_ID
- Movie_ID
- Rating

Users and movies with fewer than three rating interactions were initially filtered out.

The resulting dataset was prepared for collaborative filtering.

### 6.5 Train-Test Split

The dataset was divided into:

- Training data: 80%
- Testing data: 20%

The training data was used to train the model, while the testing data was used to evaluate prediction accuracy.

### 6.6 Model Development

The Singular Value Decomposition algorithm was implemented using the Scikit-Surprise library.

The initial model configuration included:

| Parameter | Value |
|---|---:|
| n_factors | 50 |
| n_epochs | 20 |
| lr_all | 0.005 |
| reg_all | 0.02 |
| random_state | 42 |

### 6.7 Model Training

The SVD algorithm was trained using historical user-movie rating interactions.

During training, the algorithm learned latent user and movie factors to estimate unknown ratings.

### 6.8 Model Evaluation

The model was evaluated using two statistical metrics:

**Root Mean Square Error (RMSE)**

RMSE measures the square-rooted average squared difference between actual and predicted ratings.

**Mean Absolute Error (MAE)**

MAE measures the average absolute difference between actual and predicted ratings.

### 6.9 Recommendation Generation

The trained model predicted ratings for movies that a selected user had not rated in the sampled dataset.

Movies were sorted according to their predicted ratings, and the highest-rated predictions were selected as recommendations.

# 7. Algorithm Used: Singular Value Decomposition

Singular Value Decomposition is a matrix factorization technique used in recommendation systems.

It represents the user-movie rating matrix through lower-dimensional latent factors.

The mathematical representation is:

R ≈ U × Σ × Vᵀ

Where:

- R represents the original rating matrix.
- U represents user-related latent factors.
- Σ represents singular values.
- Vᵀ represents movie-related latent factors.

The SVD implementation learns relationships between users and movies from their historical ratings.

It then uses the learned factors to estimate ratings for previously unrated movies.

# 8. Implementation and Results

The project was implemented in Google Colab using Python.

### 8.1 Data Analysis Results

The following preprocessing and analysis activities were completed:

- 200,000 rating records sampled.
- Missing values checked.
- Duplicate records checked.
- Rating distributions visualized.
- Popular movies analyzed.
- User activity analyzed.
- Yearly rating trends visualized.
- Movie genres analyzed separately.

### 8.2 Initial Model Evaluation Results

The initial SVD model produced the following results:

| Evaluation Metric | Score |
|---|---:|
| RMSE | 1.003243 |
| MAE | 0.808518 |

The MAE result indicates that the average absolute prediction error was approximately 0.81 rating points.

The RMSE result indicates that the model's root mean squared prediction error was approximately 1.00 rating points.

These results provide the baseline performance of the recommendation model.

### 8.3 Top 10 Movie Recommendations

The recommendation engine generated the following predictions for the selected user:

| Movie ID | Predicted Rating |
|---:|---:|
| 2102 | 4.565113 |
| 3046 | 4.535007 |
| 2452 | 4.511295 |
| 2057 | 4.363807 |
| 2172 | 4.323556 |
| 4306 | 4.281914 |
| 3290 | 4.236413 |
| 241 | 4.215375 |
| 1256 | 4.211903 |
| 2162 | 4.207742 |

The model ranked these movies according to their predicted ratings.

The predicted scores suggest that these movies may be preferred by the selected user based on learned rating patterns.

**Note:** These IDs have not yet been mapped to actual movie titles.

### 8.4 Optimized Model Results

The optimized model's final RMSE and MAE should be inserted after completing hyperparameter tuning.

| Evaluation Metric | Initial Model | Optimized Model |
|---|---:|---:|
| RMSE | 1.003243 | Pending |
| MAE | 0.808518 | Pending |

### 8.5 Recommendation Quality Evaluation

Precision@10 and Recall@10 are additional metrics intended to evaluate recommendation relevance.

These results will be added after completing the ranking evaluation.

# 9. Limitations of the Project

The current implementation has certain limitations:

1. The project uses a sample of 200,000 ratings rather than the entire rating dataset.
2. The recommendation system depends on historical user rating information.
3. New users may receive less personalized recommendations because of limited interaction history.
4. New movies may have insufficient rating information.
5. The supplied movie metadata and Netflix rating dataset use different movie ID systems.
6. The current evaluation focuses primarily on rating prediction accuracy.
7. The system does not yet include a graphical user interface for real-world users.

# 10. Future Scope

The project can be improved and expanded in several ways:

1. **Larger Dataset:** Train the recommendation model using a larger portion of the available rating records.
2. **Hyperparameter Optimization:** Experiment with different SVD configurations to improve prediction accuracy.
3. **Hybrid Recommendation System:** Combine collaborative filtering with content-based filtering.
4. **Movie Metadata Integration:** Integrate compatible movie titles, genres and descriptions.
5. **Recommendation Interface:** Develop a web application using Flask, Streamlit or React.
6. **Advanced Evaluation:** Use Precision@K, Recall@K and other ranking metrics.
7. **Real-Time Recommendations:** Allow users to submit ratings and receive updated recommendations.
8. **Scalability:** Explore methods for efficiently handling larger datasets and more users.

# 11. Conclusion

The Netflix Movie Recommendation Engine demonstrates how machine learning can be used to understand user preferences and generate personalized movie recommendations.

The project involved data collection, preprocessing, exploratory data analysis, feature preparation, model development and evaluation.

The Singular Value Decomposition algorithm was implemented using the Scikit-Surprise library to learn patterns from historical movie ratings and predict ratings for previously unrated movies.

The initial model achieved an RMSE of 1.003243 and an MAE of 0.808518 on the testing dataset. It also successfully generated a ranked list of ten movie recommendations for a selected user.

Overall, the project provides practical experience in data analysis, collaborative filtering and machine learning-based recommendation systems.

Further improvements, including hyperparameter optimization, compatible movie metadata integration and a user-friendly interface, can make the system more useful and scalable.

# 12. References

1. Netflix Prize Dataset – Historical movie rating dataset.
2. Scikit-Surprise Documentation – https://surpriselib.com/
3. Pandas Documentation – https://pandas.pydata.org/docs/
4. NumPy Documentation – https://numpy.org/doc/
5. Matplotlib Documentation – https://matplotlib.org/stable/
6. Seaborn Documentation – https://seaborn.pydata.org/
7. Google Colab – https://colab.research.google.com/
